# Case Study Solution — ShopEase Memory-Aware Shopping Assistant
**Based on:** `04_memory_agent.ipynb` (Short-Term Memory) applied to `Case_Study_ShopEase_Memory_Agent.md`
**Presented by:** Sharad Rajore | **Organization:** Zensar Technologies

---

### 🎯 What this notebook proves

ShopEase's product manager filed three complaints during the beta. Each one maps to a milestone below,
and each milestone reuses one of the three memory approaches from Lesson A — now wired to real
shopping tools (`search_products`, `add_to_cart`, `view_cart`, `check_order_status`,
`apply_discount_code`) instead of the weather/stock toy tools.

| # | Complaint | Root cause | Milestone that fixes it |
|---|---|---|---|
| 1 | "Actually make it size 9" was met with "What item?" | No memory at all | **M1** — manual history |
| 2 | Tester A saw tester B's cart | Memory not isolated per session | **M2** — `MemorySaver` + `thread_id` |
| 3 | A staging restart wiped every cart | Memory not durable | **M3** — `SqliteSaver` |
| — | Pilot sign-off | All of the above, live | **M4** — pilot-readiness demo |

Read this notebook top to bottom like the ShopEase engineering team would: prove the bug, patch it,
prove the patch, move to the next gap. Every code cell is preceded by a markdown cell explaining
*why* it exists, not just what it does.

## ⚙️ Setup

Same environment as Lesson A — one local/cloud LLM via `ChatOllama`, no external services required.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import langchain
print(f"LangChain: {langchain.__version__}")

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)

# ── Alternatively, use Groq ──────────────────────────────────────────────────
# from langchain_groq import ChatGroq
# llm = ChatGroq(model="llama-3.3-70b-versatile")
# ─────────────────────────────────────────────────────────────────────────────

print("LLM ready.")

---
## Section 4 — ShopEase's Mock Catalog & "Database"

Per the case study, no real database is needed: a hardcoded catalog and two plain Python dicts
(`carts`, `discounts`) stand in for ShopEase's product and cart services. Keeping these as simple
module-level dicts also makes the M2 isolation test easy to verify programmatically later — we can
inspect `carts["shopper_A"]` directly instead of just eyeballing the bot's reply.

In [ ]:
CATALOG = [
    {"name": "Blue Sneakers",       "price": 59.99,  "sizes": [7, 8, 9, 10]},
    {"name": "Red Running Shoes",   "price": 79.99,  "sizes": [8, 9, 10, 11]},
    {"name": "White Canvas Shoes",  "price": 39.99,  "sizes": [6, 7, 8, 9]},
    {"name": "Black Leather Boots", "price": 129.99, "sizes": [8, 9, 10]},
]

ORDERS = {
    "ORD1001": "Shipped - arriving Jul 5, 2026",
    "ORD1002": "Processing - ships within 2 business days",
}

DISCOUNT_CODES = {"SAVE10": 0.10, "WELCOME20": 0.20}

print(f"Catalog: {len(CATALOG)} products | Orders: {list(ORDERS)} | Codes: {list(DISCOUNT_CODES)}")

### 🔑 The one piece of plumbing the lesson's tools didn't need

`get_weather(city)` and `get_stock_price(ticker)` were stateless — every argument they needed came
from the LLM. ShopEase's `add_to_cart` / `view_cart` / `apply_discount_code` are **not** stateless:
they must read and write *this shopper's* cart, not the whole store's. But LangChain calls a tool
with only the arguments the LLM decides to pass — it won't hand the tool a `thread_id` on its own.

The fix used throughout this notebook: a tiny holder, `_current_thread_id`, that we set immediately
before every `agent.invoke()` call. The tools read it to know which shopper's cart bucket to use.
This is exactly the kind of wiring bug complaint #2 was about — get it wrong, and shopper A's tool
calls land in shopper B's cart even if the *conversation* memory is perfectly isolated.

In [ ]:
carts: dict[str, list[dict]] = {}
discounts: dict[str, float] = {}
_current_thread_id: dict[str, str] = {"value": None}

def set_current_session(thread_id: str) -> None:
    """Call this right before agent.invoke() so the cart tools know which shopper is talking."""
    _current_thread_id["value"] = thread_id

def _find_catalog_item(item_name: str):
    for item in CATALOG:
        if item_name.strip().lower() in item["name"].lower():
            return item
    return None

def _cart_total(thread_id: str) -> float:
    subtotal = sum(line["price"] for line in carts.get(thread_id, []))
    pct = discounts.get(thread_id, 0.0)
    return round(subtotal * (1 - pct), 2)

print("Session-state helpers ready.")

### 🛠️ The five ShopEase tools

Modeled directly on the lesson's `@tool`-decorated, string-returning functions. `add_to_cart` doubles
as the "change my mind on size" handler from complaint #1: if the item is already in the cart it
**updates** the size instead of adding a duplicate line.

In [ ]:
from langchain_core.tools import tool

@tool
def search_products(query: str) -> str:
    """Search the ShopEase catalog for products matching a query, e.g. 'sneakers' or 'boots'."""
    matches = [i for i in CATALOG if query.strip().lower() in i["name"].lower()]
    if not matches:
        return f"No products matched '{query}'. Try: sneakers, running shoes, canvas shoes, boots."
    lines = [
        f"- {m['name']} — ${m['price']:.2f} — sizes {', '.join(str(s) for s in m['sizes'])}"
        for m in matches[:3]
    ]
    return "Found:\n" + "\n".join(lines)


@tool
def add_to_cart(item_name: str, size: str = None) -> str:
    """Add an item to the current shopper's cart, or update its size if it's already in the cart.
    item_name should match a product from search_products. size is optional."""
    thread_id = _current_thread_id["value"]
    catalog_item = _find_catalog_item(item_name)
    if catalog_item is None:
        return f"Couldn't find '{item_name}' in the catalog. Try search_products first."

    cart = carts.setdefault(thread_id, [])
    existing = next((line for line in cart if line["item"] == catalog_item["name"]), None)
    if existing:
        existing["size"] = size or existing["size"]
        return f"Updated {catalog_item['name']} in your cart to size {existing['size']}."

    cart.append({"item": catalog_item["name"], "size": size, "price": catalog_item["price"]})
    size_note = f" (size {size})" if size else " (size not specified yet)"
    return f"Added {catalog_item['name']}{size_note} to your cart — ${catalog_item['price']:.2f}."


@tool
def view_cart() -> str:
    """Show the current shopper's cart contents and running total."""
    thread_id = _current_thread_id["value"]
    cart = carts.get(thread_id, [])
    if not cart:
        return "Your cart is empty."
    lines = [f"- {l['item']} (size {l['size'] or '?'}) — ${l['price']:.2f}" for l in cart]
    pct = discounts.get(thread_id, 0.0)
    discount_note = f"\nDiscount applied: {int(pct * 100)}%" if pct else ""
    return "Your cart:\n" + "\n".join(lines) + discount_note + f"\nTotal: ${_cart_total(thread_id):.2f}"


@tool
def check_order_status(order_id: str) -> str:
    """Look up the status of a past order by its order ID, e.g. 'ORD1001'."""
    return ORDERS.get(order_id.strip().upper(), f"No order found with ID '{order_id}'.")


@tool
def apply_discount_code(code: str) -> str:
    """Apply a discount code to the current shopper's cart."""
    thread_id = _current_thread_id["value"]
    pct = DISCOUNT_CODES.get(code.strip().upper())
    if pct is None:
        return f"'{code}' is not a valid discount code."
    discounts[thread_id] = pct
    return f"Applied code '{code.upper()}' ({int(pct * 100)}% off). New total: ${_cart_total(thread_id):.2f}."


SHOPEASE_TOOLS = [search_products, add_to_cart, view_cart, check_order_status, apply_discount_code]
print(f"Tools ready: {[t.name for t in SHOPEASE_TOOLS]}")

### 📝 System prompt

Section 4 explicitly requires the assistant to use cart/session context instead of re-asking for information it was already given — that instruction has to be spelled out, not assumed.

In [ ]:
SHOPEASE_SYSTEM_PROMPT = (
    "You are ShopEase's shopping assistant. Use the conversation history and the cart/session "
    "context you already have — never re-ask the shopper for an item, size, or detail they already "
    "gave you earlier in this conversation. If a shopper says something like 'actually make it size 9' "
    "without repeating the item name, infer which item they mean from the conversation so far and "
    "call add_to_cart again with the corrected size. Use the tools provided for search, cart, orders, "
    "and discounts."
)
print("System prompt ready.")

---
## M1 — Prove the Bug, Then Patch It Manually

**Complaint #1:** *"Add the blue sneakers to my cart." → next message: "Actually make it size 9."
Bot asked "What item are you referring to?"*

First we reproduce it with **no memory at all** (mirrors Lesson A, Part 1). Then we patch it with the
**manual history** pattern (Lesson A, Approach 1).

In [ ]:
from langchain.agents import create_agent

# No checkpointer -> no memory, exactly like the lesson's agent_no_memory
agent_no_memory = create_agent(
    model=llm,
    tools=SHOPEASE_TOOLS,
    system_prompt=SHOPEASE_SYSTEM_PROMPT,
)

print("Agent created (no memory).")

In [ ]:
set_current_session("m1_before_demo")

print("=" * 60)
print("Turn 1:")
r1 = agent_no_memory.invoke({"messages": [("user", "I'd like to add the blue sneakers to my cart.")]})
print("Shopper: I'd like to add the blue sneakers to my cart.")
print(f"Bot:     {r1['messages'][-1].content}")

print()
print("=" * 60)
print("Turn 2 (complaint #1 — shopper doesn't repeat the item name):")
r2 = agent_no_memory.invoke({"messages": [("user", "Actually make it size 9.")]})
print("Shopper: Actually make it size 9.")
print(f"Bot:     {r2['messages'][-1].content}")

print()
print("🔴 BEFORE: each invoke() is a blank slate, so 'it' means nothing to the bot.")
print(f"Cart for this session: {carts.get('m1_before_demo', [])}")

### 🔍 Before — what just happened

Turn 2's `invoke()` call only contained *that one message*. The model never saw Turn 1, so it had no
way to resolve "it", and (depending on the run) either asked a clarifying question or guessed the
wrong item. This is complaint #1, captured as "before" evidence.

### Now patch it — Approach 1: manual message history

Same agent object, same tools — the only change is *how we call it*. We keep a Python list and resend it on every turn (the lesson's `chat()` pattern), and call `set_current_session()` so the cart tools stay in sync with which shopper's cart we're building.

In [ ]:
conversation_history = []

def shopease_chat(user_message: str, thread_id: str = "m1_after_demo") -> str:
    """Manual-history chat helper — Lesson A Approach 1, adapted for ShopEase."""
    set_current_session(thread_id)
    conversation_history.append({"role": "user", "content": user_message})
    response = agent_no_memory.invoke({"messages": conversation_history})
    ai_reply = response["messages"][-1].content
    conversation_history.append({"role": "assistant", "content": ai_reply})
    return ai_reply

print("Manual shopease_chat() ready.")

In [ ]:
print("=" * 60)
print("Turn 1:")
reply1 = shopease_chat("I'd like to add the blue sneakers to my cart.")
print(f"Bot: {reply1}")

print()
print("=" * 60)
print("Turn 2 (same wording as the complaint — no item name repeated):")
reply2 = shopease_chat("Actually make it size 9.")
print(f"Bot: {reply2}")

print()
print(f"🟢 AFTER: {carts.get('m1_after_demo')}")
print("✅ The manual history let the model see 'blue sneakers' from Turn 1 when resolving 'it' in Turn 2.")

### ⚠️ Why this fix can't ship (ties back to the lesson's "Approach 1 Limitations" table)

- **We manage the list.** `conversation_history` is one global Python list — miss an `append()`, or use
  it from two request handlers at once, and the bug comes right back.
- **No multi-shopper support.** One global list means one conversation. Real traffic is hundreds of
  shoppers at once — this is complaint #2 waiting to happen.
- **No persistence.** A process restart empties the list — this is complaint #3 waiting to happen.
- **No pruning.** The list only grows. By turn 200 every call resends the whole history, so cost and
  latency creep up with no ceiling.

**Conclusion:** M1 proves the concept (send full history → memory works) but is a prototype, not a
pilot build. M2 and M3 replace the plumbing without changing that core idea.

---
## M2 — Multi-Shopper Session Isolation

**Complaint #2:** *Two beta testers used the bot at the same time from two browser tabs; one saw the
other's cart.*

Swap the manual list for `MemorySaver` + `thread_id` (Lesson A, Approach 2) — LangGraph now manages a
separate message history per `thread_id` automatically.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

shopease_memory = MemorySaver()

agent_with_memory = create_agent(
    model=llm,
    tools=SHOPEASE_TOOLS,
    system_prompt=SHOPEASE_SYSTEM_PROMPT,
    checkpointer=shopease_memory,
)

print("Agent with MemorySaver ready — one thread_id per shopper.")

In [ ]:
def shopper_turn(agent, thread_id: str, user_message: str) -> str:
    """Send one message as a given shopper and return the bot's reply.
    set_current_session() keeps the cart tools aligned with the same thread_id
    LangGraph is using to key the conversation history."""
    set_current_session(thread_id)
    config = {"configurable": {"thread_id": thread_id}}
    response = agent.invoke({"messages": [("user", user_message)]}, config=config)
    return response["messages"][-1].content

print("shopper_turn() ready.")

In [ ]:
print("Shopper A adds Red Running Shoes:")
print(" ", shopper_turn(agent_with_memory, "shopper_A", "Add the red running shoes to my cart, size 10."))

print()
print("Shopper B adds White Canvas Shoes (interleaved with A's session):")
print(" ", shopper_turn(agent_with_memory, "shopper_B", "Add the white canvas shoes to my cart, size 7."))

print()
print("Shopper A adds a second item:")
print(" ", shopper_turn(agent_with_memory, "shopper_A", "Also add the black leather boots, size 9."))

print()
print("Shopper A: what's in my cart?")
reply_A = shopper_turn(agent_with_memory, "shopper_A", "What's in my cart?")
print(" ", reply_A)

print()
print("Shopper B: what's in my cart?")
reply_B = shopper_turn(agent_with_memory, "shopper_B", "What's in my cart?")
print(" ", reply_B)

### ✅ Isolation test — asserted, not eyeballed

Exit criteria for M2 is a *programmatic* check. We verify both the ground-truth `carts` dict and the bot's own reply text never cross shoppers.

In [ ]:
cart_A = carts.get("shopper_A", [])
cart_B = carts.get("shopper_B", [])

items_A = {line["item"] for line in cart_A}
items_B = {line["item"] for line in cart_B}

assert items_A == {"Red Running Shoes", "Black Leather Boots"}, f"Shopper A cart wrong: {items_A}"
assert items_B == {"White Canvas Shoes"}, f"Shopper B cart wrong: {items_B}"
assert "White Canvas Shoes" not in items_A, "Cross-contamination: shopper A can see shopper B's item!"
assert not ({"Red Running Shoes", "Black Leather Boots"} & items_B), \
    "Cross-contamination: shopper B can see shopper A's item!"

# Same check on the bot's own words, since that's what the shopper actually reads
assert "canvas" not in reply_A.lower(), "Shopper A's reply mentions shopper B's item!"
assert "running shoes" not in reply_B.lower() and "boots" not in reply_B.lower(), \
    "Shopper B's reply mentions shopper A's item!"

print("✅ Isolation test passed — shopper A and shopper B never see each other's cart.")

### 🔬 What is `MemorySaver` actually storing?

In [ ]:
config_to_inspect = {"configurable": {"thread_id": "shopper_A"}}
checkpoint = shopease_memory.get(config_to_inspect)

print("📦 Stored messages for 'shopper_A':")
if checkpoint:
    stored_messages = checkpoint["channel_values"].get("messages", [])
    for i, msg in enumerate(stored_messages):
        role = msg.__class__.__name__.replace("Message", "")
        content = str(msg.content)[:80] + "..." if len(str(msg.content)) > 80 else str(msg.content)
        print(f"  [{i+1}] {role}: {content}")
else:
    print("  No checkpoint found.")

`shopease_memory.get(config)` returns the full LangGraph checkpoint for that one `thread_id` — the
`messages` channel, in order, including every human turn, AI reply, and the tool calls/tool results in
between. It is **not** our `carts` dict; that's a separate application-level store we built ourselves.
LangGraph only checkpoints its own graph state (here, just `messages`). That split is exactly why the
`set_current_session()` wiring matters: if it ever pointed at the wrong `thread_id`, the *conversation*
could stay perfectly isolated per shopper while the *cart* silently leaked — a subtler version of
complaint #2 that assert-based testing (not eyeballing) is designed to catch.

---
## M3 — Survive a Restart

**Complaint #3:** *The staging server restarted mid-conversation (routine deploy). Every active
shopper's cart and conversation vanished.*

`MemorySaver` lives in RAM — kill the process and it's gone. `SqliteSaver` (Lesson A, Approach 3)
writes every checkpoint to disk. To *prove* durability rather than assume it, this section actually
closes the SQLite connection and rebuilds the checkpointer + agent from scratch, simulating a real
process restart, then resumes the same `thread_id`.

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

db_path = "shopease_sessions.db"

conn = sqlite3.connect(db_path, check_same_thread=False)
sqlite_checkpointer = SqliteSaver(conn)

agent_persistent = create_agent(
    model=llm,
    tools=SHOPEASE_TOOLS,
    system_prompt=SHOPEASE_SYSTEM_PROMPT,
    checkpointer=sqlite_checkpointer,
)

print(f"Agent with SqliteSaver ready → '{db_path}'")

In [ ]:
print("=== SESSION 1 (before the 'restart') ===")
thread_id = "persistent_shopper_1"

print(shopper_turn(agent_persistent, thread_id, "Hi, I'm building a cart. Add the black leather boots, size 9."))
print(shopper_turn(agent_persistent, thread_id, "Apply the code SAVE10."))
print(shopper_turn(agent_persistent, thread_id, "What's in my cart now?"))

In [ ]:
print("💥 Simulating a routine deploy / server restart (complaint #3)...")

conn.close()
del sqlite_checkpointer, agent_persistent
print("Closed the SQLite connection and dropped the agent object — nothing is held in RAM anymore.")

conn2 = sqlite3.connect(db_path, check_same_thread=False)
sqlite_checkpointer_2 = SqliteSaver(conn2)

agent_persistent_2 = create_agent(
    model=llm,
    tools=SHOPEASE_TOOLS,
    system_prompt=SHOPEASE_SYSTEM_PROMPT,
    checkpointer=sqlite_checkpointer_2,
)

print("Reopened the connection and rebuilt the agent from scratch — this is 'Session 2'.")

In [ ]:
print("=== SESSION 2 (resuming the SAME thread_id after the 'restart') ===")
reply = shopper_turn(agent_persistent_2, thread_id, "What's in my cart, and did my discount code stick?")
print(reply)

assert "boot" in reply.lower(), "Cart did not survive the restart!"
print()
print("✅ The bot recalled the boots and the discount from Session 1 — the message history was on disk.")

In [ ]:
import os

db_size = os.path.getsize(db_path)
print(f"Database file: {db_path}")
print(f"Size on disk:  {db_size} bytes")
print()
print("This file — not the Python process — is what makes M3 different from M2.")

### ⚠️ An honest gap in this solution (worth naming, not hiding)

`SqliteSaver` persists the **conversation** — the `messages` channel LangGraph checkpoints. It does
**not** persist our `carts` / `discounts` dicts; those are plain Python objects living in the
notebook's process, per Section 4's "a Python dict is fine" instruction. They "survived" the restart
above only because we never actually killed the Python kernel — we tore down and rebuilt the
*checkpointer and SQLite connection*, which is the specific thing M3 asks us to prove.

A real production restart (a new pod, a new process) would still wipe `carts` unless it also moves to
real storage (Postgres/Redis/etc). Worse: the LLM would still *say* the boots are in the cart — because
that's what the persisted conversation says — even if the cart tool itself had actually forgotten. This
mismatch is the honest limitation carried into M4.

---
## M4 — Pilot-Readiness Demo

Framed exactly as it would be presented to the ShopEase product manager who filed the three
complaints: restate them as resolved, then run one live, continuous shopper journey on the
**production-configured agent** (`agent_persistent_2`, the `SqliteSaver`-backed one from M3).

In [ ]:
demo_thread = "pilot_demo_shopper"

print("🎬 ShopEase pilot-readiness demo — one continuous shopper journey\n")

steps = [
    "Hi, I'm looking for something to wear for a run.",
    "Add the red running shoes to my cart, size 10.",
    "Actually, make that size 11.",
    "Apply the discount code WELCOME20.",
]

for msg in steps:
    print(f"Shopper: {msg}")
    print(f"Bot:     {shopper_turn(agent_persistent_2, demo_thread, msg)}")
    print()

print("— [Shopper closes the browser tab] —")
print("— [Comes back later; the app looks up their saved thread_id from their login session] —\n")

closing_msg = "What's in my cart, and what's the status of order ORD1001?"
print(f"Shopper: {closing_msg}")
print(f"Bot:     {shopper_turn(agent_persistent_2, demo_thread, closing_msg)}")

### Which memory approach powers production?

**`SqliteSaver` (M3)** — the exact `agent_persistent_2` object used above — because it's the only one
of the three that survives a deploy, a crash, or a pod restart, which is the pilot's explicit
condition for approval.

- **M1 (manual list)** is dev-only: one global list, no isolation, nothing durable. Fine for proving
  the *concept* of memory; unacceptable the moment two shoppers exist at once.
- **M2 (`MemorySaver`)** is a real step up — cleanly isolated per `thread_id` — but still lives in RAM.
  That's the exact failure the beta already hit (complaint #3, the staging restart).

### Complaints — resolved

1. ✅ *"Actually make it size 9"* — history (manual or graph-managed) means the model sees the earlier turn.
2. ✅ *Two tabs, two carts* — `thread_id` isolation, proven with asserts in M2.
3. ✅ *Staging restart mid-conversation* — `SqliteSaver`, proven by literally closing and reopening the
   connection in M3.

### One honest limitation

Only the *conversation* is durable today — the cart itself is an in-memory dict (see the M3 caveat).
Before a real pilot, cart state needs to move to real storage, or be reconstructed from the persisted
tool-call history on startup. Further out: a shopper who returns **next week** in a brand-new
`thread_id` still gets a blank slate even with `SqliteSaver` — solving that needs a second,
`user_id`-scoped store (LangGraph's `store`, as distinct from `checkpointer`), which is explicitly out
of scope for this build (Section 7 of the case study) and is named here as the next step, not solved.

---
## 📋 Non-Functional Write-Up (Section 6)

*A sentence or two on each point, specific to ShopEase — not generic textbook answers.*

**Session ID strategy.** Don't let the client hand-pick `thread_id`. Derive it server-side from
something already authenticated: the logged-in shopper's session cookie / auth token for signed-in
users, or a server-issued anonymous session ID (HttpOnly cookie) for guests. The browser never gets to
choose its own `thread_id` — that's the actual difference between "my cart" and "anyone's cart."

**Context growth.** Every turn resends the full message list to the LLM, so cost and latency grow
roughly linearly with turn count; by turn 200 a session is slow, expensive, and eventually overflows
the context window. Mitigation: once a thread crosses a turn/token threshold, trim or summarize older
turns (keep the last N verbatim plus a running summary of everything before) — not implemented here,
flagged as the next hardening step before general availability.

**PII handling.** Names, shipping addresses, and order history will flow through this same checkpoint
once real tools are wired up. A checkpoint that never expires is an unbounded PII store by accident.
Apply a retention/TTL policy to the checkpoint table (purge threads N days after their last turn), and
never let payment details land in the conversation text itself, even transiently.

**Failure mode.** If `SqliteSaver` can't reach the `.db` file (disk full, permissions, a locked file
mid-migration), the request must not crash. Catch the storage error, tell the shopper something like
*"I'm having trouble saving your session right now — you can keep browsing, but please don't close this
tab,"* and fall back to an in-memory session for that request so they aren't blocked, while alerting
on-call that persistence is degraded.

**Concurrency.** Two tabs, same shopper, same `thread_id`, sending messages near-simultaneously is a
known limitation, not something M1–M4 solve. LangGraph checkpoints the whole state per turn, so two
interleaved writes can race and one tab's update can silently overwrite the other's (e.g. two
"add to cart" calls landing on stale state). For a low-traffic pilot this is an acceptable named risk;
before general availability it needs per-thread request serialization or optimistic-concurrency
checks on the checkpoint version.

---
## ✅ Deliverables Checklist

- [x] Working notebook implementing M1–M3, each in its own clearly labeled section (above)
- [x] Before/after transcript for M1 (`m1-before-demo`, `m1-after-demo` cells)
- [x] Isolation test output for M2 (`m2-assert` cell — asserts, not just prints)
- [x] Restart-survival proof for M3 (`m3-teardown` / `m3-session2` cells, plus `shopease_sessions.db` on disk)
- [x] One-page write-up covering Section 6 (`nfr-header` cell, above)
- [x] Live demo script for M4 (`m4-demo` cell) — run it end-to-end as the 5-minute stakeholder walkthrough

---
## 📌 Summary

| Milestone | Memory approach | Complaint it resolves | Ships to production? |
|---|---|---|---|
| M1 | Manual `conversation_history` list | #1 — forgets mid-conversation | ❌ prototype only |
| M2 | `MemorySaver` + `thread_id` | #2 — cross-shopper leakage | ⚠️ dev/single-server only |
| M3 | `SqliteSaver` | #3 — restart wipes state | ✅ pilot-ready |
| M4 | M3, reused end-to-end | All three, live | — this is the demo, not a new mechanism |

**Root cause, restated:** an LLM is stateless; "memory" is just resending history on every call. The
three milestones only ever change *who manages that history and where it lives* — never the underlying
idea. **Named next step (Section 7, not built here):** a `user_id`-scoped `store`, separate from the
`thread_id`-scoped `checkpointer`, for shoppers who return in a brand-new session.